# # Main
[add a description/summary/abstract here later]

## 1. Setup
Install all required libraries and/or set up the environment.

Note: these older Numpy and Torch versions are needed otherwise Recbole (v1.2.1) breaks

In [ ]:
!python --version

Python 3.11.13


In [ ]:
!pip list

Package                               Version
------------------------------------- -------------------
absl-py                               1.4.0
accelerate                            1.8.1
aiofiles                              24.1.0
aiohappyeyeballs                      2.6.1
aiohttp                               3.11.15
aiosignal                             1.4.0
alabaster                             1.0.0
albucore                              0.0.24
albumentations                        2.0.8
ale-py                                0.11.2
altair                                5.5.0
annotated-types                       0.7.0
antlr4-python3-runtime                4.9.3
anyio                                 4.9.0
argon2-cffi                           25.1.0
argon2-cffi-bindings                  21.2.0
array_record                          0.7.2
arviz                                 0.21.0
astropy                               7.1.0
astropy-iers-data                     0.2025.7.14.0.

In [8]:
# Core packages
!pip install -U pip
!pip install numpy==1.26.4
!pip install pandas==2.3.1
!pip install torch==2.5.1
!pip install scikit-learn==1.7.1
!pip install matplotlib==3.10.3

# Recommender system packages
!pip install lenskit==2025.1.1
!pip install recbole==1.2.1

### 1.1 Drive
Load Google Drive and copy the datasets folder

In [9]:
import os
import shutil
from google.colab import drive


drive.mount("/content/drive", readonly=True)
source = "/content/drive/My Drive/Workspace/Thesis/Repository/unreasonable-effectiveness-recsys/datasets"
destination = "./datasets"

shutil.copytree(source, destination)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileExistsError: [Errno 17] File exists: './datasets'

## 2. Parameters
Global variables and shared parameters

### 2.1 Experiment variables
Fixed experiment parameters

In [11]:

RECOMMENDATIONS = 10
SEED = 42
SIZES = [i * 0.1 for i in range(1, 11)] # 0.1, 0.2, ..., 1.0
COLUMN_NAMES = {"user_id": "user_id", "item_id": "item_id", "rating": "rating"}


### 2.2 Hyperparameters
Shared hyperparameters

In [14]:

TEST_SIZE = 0.2
PARTITIONS = 5 # For cross-validation
KNN_K = 20
FEATURES = 50
TRAINING_RECOMMENDATIONS = 100


## 3. Load
Load the desired dataset as a Pandas DataFrame of user ID, item ID, and rating in that order

In [4]:
from enum import Enum
import pandas as pd


class Datasets(Enum):
  MOVIELENS = "movielens"
  GOODREADS = "goodreads"
  AMAZON = "amazon"
  MOVIETWEETINGS = "movietweetings"
  PERSONALITY = "personality"

def load(dataset: Datasets = Datasets.MOVIELENS) -> pd.DataFrame:
  path = f"./datasets/{dataset.value}/ratings.csv"
  df = pd.read_csv(path, sep=',', usecols=[0, 1, 2], names=[
    COLUMN_NAMES["user_id"],
    COLUMN_NAMES["item_id"],
    COLUMN_NAMES["rating"]
  ])
  df = df.reset_index(drop=True)
  return df


## 4. Sample
Take a sample of size N within the globally defined range

In [5]:
import pandas as pd


def sample(df: pd.DataFrame, size: float = SIZES[(len(SIZES) // 2) - 1]) -> pd.DataFrame:
  if size not in SIZES:
    raise ValueError(f"Size must be one of '{SIZES}'")

  size = int(size * len(df))
  return df.sample(size, random_state=SEED)


## 5. Splitting (unused)
Data-frame splitting

### 5.1. Horizontal split (unused)
Split the data-frame into training and testing data-frames

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


def horizontal_split(df: pd.DataFrame, test_size: float = TEST_SIZE):
  return train_test_split(df, test_size=test_size, random_state=SEED)


### 5.2. Vertical split (unused)
Split the training and testing data into X and y data-frames (features and target)

In [ ]:
import pandas as pd


def vertical_split(train: pd.DataFrame, test: pd.DataFrame):
  X_train, X_test = train.drop(columns=["rating"]), test.drop(columns=["rating"])
  y_train, y_test = train["rating"], test["rating"]
  return X_train, X_test, y_train, y_test


## 6. LensKit
Use LensKit to train and evaluate a given data-frame returning a single float of the mean NDCG

In [21]:
from enum import Enum
import pandas as pd
from lenskit.data import from_interactions_df, UserIDKey, ItemListCollection
from lenskit.splitting import SampleFrac, crossfold_users
from lenskit.knn import ItemKNNScorer
from lenskit.als import BiasedMFScorer
from lenskit.pipeline import topn_pipeline
from lenskit.batch import recommend
from lenskit.metrics import RunAnalysis, NDCG


class Scorer(Enum):
  ITEM_KNN = 0
  BIASED_MF = 1

SCORERS = {
  Scorer.ITEM_KNN.name: lambda: ItemKNNScorer(k=KNN_K),
  Scorer.BIASED_MF.name: lambda: BiasedMFScorer(features=FEATURES),
}

def use_lenskit(df: pd.DataFrame, scorer: Scorer = Scorer.ITEM_KNN) -> float:
  dataset = from_interactions_df(
    df,
    user_col=COLUMN_NAMES["user_id"],
    item_col=COLUMN_NAMES["item_id"],
    rating_col=COLUMN_NAMES["rating"],
    timestamp_col=None,
  )

  get = SCORERS[scorer.name]
  model = get()
  pipe = topn_pipeline(model)

  all_test = ItemListCollection(key=UserIDKey)
  all_recs = ItemListCollection(["model", "user_id"])

  for split in crossfold_users(dataset, PARTITIONS, SampleFrac(TEST_SIZE)):
    all_test.add_from(split.test)
    fit = pipe.clone()
    fit.train(split.train)
    recs = recommend(fit, split.test.keys(), n=TRAINING_RECOMMENDATIONS)
    all_recs.add_from(recs, model=f"Only")

  ran = RunAnalysis()
  ran.add_metric(NDCG(k=RECOMMENDATIONS))
  results = ran.measure(all_recs, all_test)
  result = results.list_summary().iloc[0].iloc[0]

  return float(result)


## 7. RecBole
Use RecBole to train and evaluate a given data-frame returning a single float of the mean NDCG

### 7.1. Parameters
Recbole specific parameters and shared variables

In [22]:
ROOT_DIRECTORY = "./recbole/"
DATASETS_DIRECTORY = ROOT_DIRECTORY + "datasets/"
DATASET_TEMP_NAME = "temp"
DATASET_TEMP_DIRECTORY = DATASETS_DIRECTORY + f"{DATASET_TEMP_NAME}/"
TEMP_ATOMIC_FILE_NAME = f"{DATASET_TEMP_NAME}.inter"
TEMP_ATOMIC_FILE_PATH = DATASET_TEMP_DIRECTORY + TEMP_ATOMIC_FILE_NAME

RECBOLE_CONFIG = {
  # Environment
  "seed": SEED,
  "data_path": DATASETS_DIRECTORY,
  "checkpoint_dir": ROOT_DIRECTORY + "saved/",

  # Data
  "field_separator": ',',
  "seq_separator": '\n',
  "USER_ID_FIELD": COLUMN_NAMES["user_id"],
  "ITEM_ID_FIELD": COLUMN_NAMES["item_id"],
  "RATING_FIELD": COLUMN_NAMES["rating"],
  "TIME_FIELD": None,
  "load_col": {"inter": [COLUMN_NAMES["user_id"], COLUMN_NAMES["item_id"], COLUMN_NAMES["rating"]]},

  # Training
  ##########

  # Evaluation
  "eval_args": {
    "split": {"RS": [10 - (TEST_SIZE * 10), 0, (TEST_SIZE * 10)]},
  },
  "metrics": ["NDCG"],
  "topk": RECOMMENDATIONS,
}

### 7.2. Prepare
Save the interactions data-frame as an "Atomic" (CSV-like) .inter file

In [23]:
import pandas as pd
import os


def save_as_atomic(df: pd.DataFrame) -> None:
  os.makedirs(os.path.dirname(TEMP_ATOMIC_FILE_PATH), exist_ok=True)
  with open(TEMP_ATOMIC_FILE_PATH, "w") as f:
    features = [f"{name}:float" if name == COLUMN_NAMES["rating"] else f"{name}:token" for name in df.columns]
    f.write(','.join(features) + '\n')
  df.to_csv(TEMP_ATOMIC_FILE_PATH, mode="a", index=False, header=False)

### 7.3. Run
Create the atomic file, call "run_recbole" using the config dictionary, and returns the NDCG@[RECOMMENDATIONS] as well

In [24]:
from enum import Enum
import pandas as pd
from recbole.quick_start import run_recbole


class Model(Enum):
  ITEM_KNN = "ItemKNN" # Item K-Nearest Neighbours
  # SPECTRAL_CF = "SpectralCF" # Spectral Collaborative Filtering
  POP = "Pop" # Popularity

def use_recbole(df: pd.DataFrame, model: Model = Model.ITEM_KNN) -> float:
  save_as_atomic(df)
  result = run_recbole(model=model.value, dataset=DATASET_TEMP_NAME, config_dict=RECBOLE_CONFIG)
  test_result = result["test_result"]
  for key, value in test_result.items():
    if "ndcg" in key.lower():
      return float(value)
  raise ValueError("No NDCG metric found")


## 8. Plotting
Graph plotting and vizualisation of results

### 8.1. Types
Type definitions for shared types

In [25]:
from typing import Dict, TypeAlias
# from pydantic import BaseModel, ValidationError

# d[library][algorithm][dataset][size] -> float
Results: TypeAlias = Dict[str, Dict[str, Dict[str, Dict[str, float]]]]

### 8.2. Prepare
Prepare and return a dictionary that would hold the results

In [26]:

# Dictionary so that 'd[library][algorithm][dataset][size] -> float' holds the NDCG
def prepare_results(sizes = SIZES[:3], libraries = ["LensKit", "RecBole"]) -> Results:
  results = {
    library: {
      algorithm.name: {
        dataset.name: {
          str(size): float() for size in sizes
        } for dataset in Datasets
      } for algorithm in (Scorer if library == "LensKit" else Model)
    } for library in libraries
  }
  return results


### 8.3. Plot results
Handle a given 'Results' dictionary instance

In [28]:
import matplotlib.pyplot as plt


MARKERS = ['o', '^', '2', 's', 'P', '*', 'X', 'D', '|']
COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple",
          "tab:brown", "tab:pink", "tab:gray", "tab:olive", "tab:cyan"]

def plot_results(results: Results, figsize=(10, 12)) -> None:
  # Setup
  libraries = list(results.keys())
  fig, axes = plt.subplots(len(libraries), 1, figsize=figsize)

  # If only one library, make axes iterable
  if len(libraries) == 1:
    axes = [axes]

  for library_index, library in enumerate(libraries):
    ax = axes[library_index]

    algorithms = list(results[library].keys())
    datasets = list(results[library][algorithms[0]].keys())  # Get datasets from first algorithm

    # Plot each algorithm-dataset combination
    for algorithm_index, algorithm in enumerate(algorithms):
      for dataset_index, dataset in enumerate(datasets):
        # Extract sizes and values
        size_value = results[library][algorithm][dataset]
        sizes = list(size_value.keys())
        # sizes = [f"{int(size * 100)}%" for size in sizes]
        values = list(size_value.values())

        marker = MARKERS[algorithm_index]
        color = COLORS[dataset_index]

        ax.plot(
          sizes, values,
          color=color,
          marker=marker,
          linewidth=2,
          markersize=5,
          label=f'{algorithm.capitalize()} - {dataset.capitalize()}',
        )

        # Customize subplot
        ax.set_title(library, fontsize=14, fontweight="bold")
        ax.set_xlabel("Dataset Size", fontsize=12)
        ax.set_ylabel(f"NDCG@{RECOMMENDATIONS}", fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

        # Set x-axis to log scale if sizes vary widely
        # if max(sizes) / min(sizes) > 10:
        #     ax.set_xscale("log")

  plt.tight_layout()
  plt.show()


# Example usage with sample data
def create_sample_data():
    """Create sample data matching your structure"""
    return {
        "LensKit": {
            "ItemKNN": {
                "MovieLens": {"1000": 0.75, "2000": 0.78, "3000": 0.80, "4000": 0.82, "5000": 0.84, "6000": 0.85, "7000": 0.86, "8000": 0.87, "9000": 0.88, "10000": 0.89},
                "Amazon": {"1000": 0.68, "2000": 0.71, "3000": 0.73, "4000": 0.75, "5000": 0.77, "6000": 0.78, "7000": 0.79, "8000": 0.80, "9000": 0.81, "10000": 0.83},
                "Netflix": {"1000": 0.72, "2000": 0.75, "3000": 0.77, "4000": 0.79, "5000": 0.81, "6000": 0.82, "7000": 0.83, "8000": 0.84, "9000": 0.85, "10000": 0.86}
            },
            "MatrixFactorization": {
                "MovieLens": {"1000": 0.73, "2000": 0.76, "3000": 0.78, "4000": 0.80, "5000": 0.82, "6000": 0.83, "7000": 0.84, "8000": 0.85, "9000": 0.86, "10000": 0.87},
                "Amazon": {"1000": 0.66, "2000": 0.69, "3000": 0.71, "4000": 0.73, "5000": 0.75, "6000": 0.76, "7000": 0.77, "8000": 0.78, "9000": 0.79, "10000": 0.81},
                "Netflix": {"1000": 0.70, "2000": 0.73, "3000": 0.75, "4000": 0.77, "5000": 0.79, "6000": 0.80, "7000": 0.81, "8000": 0.82, "9000": 0.83, "10000": 0.84}
            }
        },
        "RecBole": {
            "ItemKNN": {
                "MovieLens": {"1000": 0.76, "2000": 0.79, "3000": 0.81, "4000": 0.83, "5000": 0.85, "6000": 0.86, "7000": 0.87, "8000": 0.88, "9000": 0.89, "10000": 0.90},
                "Amazon": {"1000": 0.69, "2000": 0.72, "3000": 0.74, "4000": 0.76, "5000": 0.78, "6000": 0.79, "7000": 0.80, "8000": 0.81, "9000": 0.82, "10000": 0.84},
                "Netflix": {"1000": 0.73, "2000": 0.76, "3000": 0.78, "4000": 0.80, "5000": 0.82, "6000": 0.83, "7000": 0.84, "8000": 0.85, "9000": 0.86, "10000": 0.87}
            },
            "FactorMatrixization": {
                "MovieLens": {"1000": 0.74, "2000": 0.77, "3000": 0.79, "4000": 0.81, "5000": 0.83, "6000": 0.84, "7000": 0.85, "8000": 0.86, "9000": 0.87, "10000": 0.88},
                "Amazon": {"1000": 0.67, "2000": 0.70, "3000": 0.72, "4000": 0.74, "5000": 0.76, "6000": 0.77, "7000": 0.78, "8000": 0.79, "9000": 0.80, "10000": 0.82},
                "Netflix": {"1000": 0.71, "2000": 0.74, "3000": 0.76, "4000": 0.78, "5000": 0.80, "6000": 0.81, "7000": 0.82, "8000": 0.83, "9000": 0.84, "10000": 0.85}
            }
        }
    }


## 9. Run
Execute using the above function and variable definitions

In [ ]:

lenskit = "LensKit"
recbole = "RecBole"
sizes = SIZES[:2]
results = prepare_results(sizes)

for dataset in Datasets:
  full = load(dataset)
  for size in sizes:
    sampled = sample(full, size)
    percentage = f"{int(size * 100)}%"

    # LensKit
    for scorer in Scorer:
      result = use_lenskit(sampled, scorer)
      results[lenskit][scorer.name][dataset.name][percentage] = result

    # RecBole
    for model in Model:
      result = use_recbole(sampled, model)
      results[recbole][model.name][dataset.name][percentage] = result

plot_results(results)


/usr/local/lib/python3.11/dist-packages/lenskit/splitting/users.py:197: DataWarning: item list collection has empty lists, they will be dropped
  test_tbl = test.to_df()[["user_id", "item_id"]]
/usr/local/lib/python3.11/dist-packages/lenskit/splitting/users.py:197: DataWarning: item list collection has empty lists, they will be dropped
  test_tbl = test.to_df()[["user_id", "item_id"]]
/usr/local/lib/python3.11/dist-packages/lenskit/splitting/users.py:197: DataWarning: item list collection has empty lists, they will be dropped
  test_tbl = test.to_df()[["user_id", "item_id"]]
/usr/local/lib/python3.11/dist-packages/lenskit/splitting/users.py:197: DataWarning: item list collection has empty lists, they will be dropped
  test_tbl = test.to_df()[["user_id", "item_id"]]
/usr/local/lib/python3.11/dist-packages/lenskit/splitting/users.py:197: DataWarning: item list collection has empty lists, they will be dropped
  test_tbl = test.to_df()[["user_id", "item_id"]]
/usr/local/lib/python3.11/dist